## 4. Data Cleaning and Alignment — Practical Implementation with Data

In this section, the objective is to transform the raw datasets into a clean and unified dataset called `master_df`.

The workflow follows five steps:

- Date parsing
- Missing value treatment
- Type conversion
- Merge and alignment
- Construction of the final master dataset

### Expanded Macroeconomic Variables

While the first stage of the project focused primarily on the EUR/USD exchange rate, sovereign yields, and yield spreads, this new stage of the research will place greater emphasis on a broader set of macroeconomic variables capable of capturing the relative economic dynamics between the United States and the Euro Area.

The objective is to progressively enrich the Machine Learning framework by incorporating indicators that reflect differences in growth conditions, inflation dynamics, labor market conditions, and macroeconomic expectations between the two economies.

A particular focus will therefore be placed on relative indicators, since foreign exchange markets are fundamentally driven by comparative economic performance rather than isolated domestic conditions.

---

## Relative Growth Indicators

The project will focus on highly comparable indicators of economic activity between the two economies.

Potential variables include:

- **GDP Growth Differential**

$$
\text{US GDP Growth} \; vs \; \text{Euro Area GDP Growth}
$$

- **Manufacturing Activity Differential**

$$
\text{US Manufacturing PMI} \; vs \; \text{Euro Area Manufacturing PMI}
$$

- **Services Activity Differential**

$$
\text{US Services PMI} \; vs \; \text{Euro Area Services PMI}
$$

- **Business Activity Differential**

$$
\text{ISM Indicators} \; vs \; \text{European Business Indicators}
$$

- **Consumption Differential**

$$
\text{US Retail Sales} \; vs \; \text{Euro Area Retail Sales}
$$

- **Industrial Production Differential**

$$
\text{US Industrial Production} \; vs \; \text{Euro Area Industrial Production}
$$

- **Consumer Confidence Differential**

$$
\text{US Consumer Confidence} \; vs \; \text{Euro Area Consumer Confidence}
$$

- **Labor Market Differential**

$$
\text{US Nonfarm Payrolls and Employment Indicators}
\; vs \;
\text{Euro Area Employment Indicators}
$$

- **Unemployment Differential**

$$
\text{US Unemployment Rate} \; vs \; \text{Euro Area Unemployment Rate}
$$

These variables aim to capture the relative strength of economic growth and business activity between the two regions. In foreign exchange markets, stronger relative economic performance generally supports the appreciation of the corresponding currency through higher investment attractiveness, stronger capital inflows, and tighter monetary policy expectations.

The use of differential indicators is particularly important because financial markets typically react not to absolute economic levels, but rather to divergences between economies.

---

## Relative Inflation Indicators

Inflation dynamics play a central role in shaping monetary policy expectations and yield differentials.

Potential variables include:

- **Inflation Differential**

$$
\text{US CPI} \; vs \; \text{Euro Area HICP}
$$

- **Core Inflation Differential**

$$
\text{US Core CPI} \; vs \; \text{Euro Area Core HICP}
$$

- **Monetary Inflation Differential**

$$
\text{US Core PCE Inflation} \; vs \; \text{Euro Area Core Inflation}
$$

- **Wage Inflation Differential**

$$
\text{US Wage Growth} \; vs \; \text{Euro Area Wage Growth}
$$

- **Inflation Expectations Differential**

$$
\text{US Inflation Expectations}
\; vs \;
\text{Euro Area Inflation Expectations}
$$

- **Energy Inflation Sensitivity**

$$
\text{Relative impact of energy prices on both economies}
$$

These indicators are intended to measure relative inflationary pressures between the United States and the Euro Area. Since central banks closely monitor inflation when determining monetary policy, inflation differentials can significantly influence interest rate expectations, sovereign yields, and ultimately exchange rate dynamics.

Higher relative inflation in one economy may lead markets to anticipate tighter monetary policy, potentially strengthening the corresponding currency through higher expected returns on financial assets denominated in that currency.

Special attention will also be given to energy-related inflation effects, particularly because the Euro Area has historically shown greater sensitivity to external energy shocks compared to the United States.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from scripts.utils import parse_dataset_dates
from scripts.utils import print_datasets_head

## 4.1 Date Parsing

In [2]:
#load datasets

eurusd = pd.read_csv("../data/eurusd.csv", sep="\t") 
eurusd.head(4)
eurusd = eurusd.reset_index()
eurusd = eurusd.iloc[:, [0, 4]]
eurusd.columns = ["Date", "EURUSD"]
eurusd["Date"] = pd.to_datetime(eurusd["Date"]).dt.strftime("%Y-%m-%d")
eurusd = eurusd[eurusd["Date"] >= "2016-01-01"]


# Growth Indicators datasets

gdp_us = pd.read_csv("../data/GDP_Growth_US.csv")
gdp_eur = pd.read_csv("../data/GDP_Growth_EUR.csv")

pmi_us = pd.read_excel("../data/PMI_US.xlsx")
pmi_eur = pd.read_csv("../data/PMI_Euro.csv")


sales_us = pd.read_csv("../data/Sales_US.csv", encoding="latin1")
sales_eur = pd.read_csv("../data/Sales_EUR_2023.csv")



ip_us = pd.read_csv("../data/Industrial_Production_US.csv",  encoding="latin1")
ip_eur = pd.read_csv("../data/Industrial_Production_EUR.csv")

conf_us = pd.read_csv("../data/Consumer_Confidence_US.csv")
conf_eur = pd.read_csv("../data/Consumer_Confidence_EUR.csv")


# Labor market Indicators datasets

nfp_us = pd.read_csv("../data/Labor_US_Nonfarm.csv",  encoding="latin1")
nfp_eur = pd.read_excel("../data/Labor_EUR_Equi_Nonfarm.xlsx")

unemp_us = pd.read_csv("../data/Unemployment_US.csv")
unemp_eur = pd.read_excel("../data/Unemployment_EUR.xlsx")

wages_us = pd.read_csv("../data/Wages_AHETPI_US.csv")
wages_eur = pd.read_csv("../data/Wages_Negotated_Wages_Eur.csv")



# Inflation Indicators datasets

cpi_us = pd.read_csv("../data/Inf_CPI_US.csv", encoding="latin1")
cpi_eur = pd.read_csv("../data/Inf_CPI_Eur.csv",encoding="latin1")

core_cpi_us = pd.read_csv("../data/Inf_Core_CPI_US.csv")
core_cpi_eur = pd.read_csv("../data/Inf_Core_CPI_Eur.csv")


In [3]:
# ==================================================
# APPLY DATE PARSING FUNCTION TO ALL DATASETS
# ==================================================

datasets = {

    # Growth Indicators
    "gdp_us": gdp_us,
    "gdp_eur": gdp_eur,

    "pmi_us": pmi_us,
    "pmi_eur": pmi_eur,

    "sales_us": sales_us,
    "sales_eur": sales_eur,

    "ip_us": ip_us,
    "ip_eur": ip_eur,

    "conf_us": conf_us,
    "conf_eur": conf_eur,

    # Labor Market Indicators
    "nfp_us": nfp_us,
    "nfp_eur": nfp_eur,

    "unemp_us": unemp_us,
    "unemp_eur": unemp_eur,

    "wages_us": wages_us,
    "wages_eur": wages_eur,

    # Inflation Indicators
    "cpi_us": cpi_us,
    "cpi_eur": cpi_eur,

    "core_cpi_us": core_cpi_us,
    "core_cpi_eur": core_cpi_eur

}


# ==================================================
# APPLY FUNCTION IN LOOP WITH ERROR CHECK
# ==================================================

cleaned_datasets = {}

for name, df in datasets.items():

    try:
        cleaned_datasets[name] = parse_dataset_dates(df)
        print(f"✅ {name} cleaned successfully")

    except Exception as e:
        print(f"❌ Error with dataset: {name}")
        print(f"Columns: {df.columns.tolist()}")
        print(e)
        print("-" * 80)

# ==================================================
# REASSIGN DATASETS
# ==================================================

gdp_us = datasets["gdp_us"]
gdp_eur = datasets["gdp_eur"]

pmi_us = datasets["pmi_us"]
pmi_eur = datasets["pmi_eur"]

sales_us = datasets["sales_us"]
sales_eur = datasets["sales_eur"]

ip_us = datasets["ip_us"]
ip_eur = datasets["ip_eur"]

conf_us = datasets["conf_us"]
conf_eur = datasets["conf_eur"]

nfp_us = datasets["nfp_us"]
nfp_eur = datasets["nfp_eur"]

unemp_us = datasets["unemp_us"]
unemp_eur = datasets["unemp_eur"]

wages_us = datasets["wages_us"]
wages_eur = datasets["wages_eur"]

cpi_us = datasets["cpi_us"]
cpi_eur = datasets["cpi_eur"]

core_cpi_us = datasets["core_cpi_us"]
core_cpi_eur = datasets["core_cpi_eur"]

✅ gdp_us cleaned successfully
✅ gdp_eur cleaned successfully
✅ pmi_us cleaned successfully
✅ pmi_eur cleaned successfully
✅ sales_us cleaned successfully
✅ sales_eur cleaned successfully
✅ ip_us cleaned successfully
✅ ip_eur cleaned successfully
✅ conf_us cleaned successfully
✅ conf_eur cleaned successfully
✅ nfp_us cleaned successfully
✅ nfp_eur cleaned successfully
✅ unemp_us cleaned successfully
✅ unemp_eur cleaned successfully
✅ wages_us cleaned successfully
✅ wages_eur cleaned successfully
✅ cpi_us cleaned successfully
✅ cpi_eur cleaned successfully
✅ core_cpi_us cleaned successfully
✅ core_cpi_eur cleaned successfully



## 4.2 Missing Values, Frequency Alignment and Resampling


In [4]:
gdp_us.columns = ["Date", "GDP Level - US"]

# Convert date
gdp_us["Date"] = pd.to_datetime(gdp_us["Date"])

# Convert values
gdp_us["GDP Level - US"] = (
    gdp_us["GDP Level - US"]
    .astype(float)
)

# Set index
gdp_us.set_index("Date", inplace=True)

# Compute quarterly growth %
gdp_us["GDP Growth - US"] = (
    gdp_us["GDP Level - US"]
    .pct_change() * 100
)

# Annualize growth to match macro convention
gdp_us["GDP Growth - US"] = (
    gdp_us["GDP Growth - US"] * 4
)

# Keep only growth column
gdp_us = gdp_us[["GDP Growth - US"]]

# Remove first row (2015 observation)
gdp_us = gdp_us.iloc[1:]

print(gdp_us.head())


            GDP Growth - US
Date                       
2016-01-01         1.970064
2016-04-01         4.011004
2016-07-01         3.867890
2016-10-01         4.165432
2017-01-01         3.996044


In [5]:
gdp_eur.columns = ["Date", "GDP Level - EUR"]

# Convert date
gdp_eur["Date"] = pd.to_datetime(gdp_eur["Date"])

# Convert GDP values
gdp_eur["GDP Level - EUR"] = (
    gdp_eur["GDP Level - EUR"]
    .astype(float)
)

# Set Date as index
gdp_eur.set_index("Date", inplace=True)

# Compute quarterly growth %
gdp_eur["GDP Growth QoQ - EUR"] = (
    gdp_eur["GDP Level - EUR"]
    .pct_change() * 100
)

# Keep only growth column
gdp_eur = gdp_eur[["GDP Growth QoQ - EUR"]]

# Remove first row (2015 data used only for calculation)
gdp_eur = gdp_eur.iloc[1:]

print(gdp_eur.head())

            GDP Growth QoQ - EUR
Date                            
2016-01-01              0.465750
2016-04-01              0.196187
2016-07-01              0.481482
2016-10-01              0.764195
2017-01-01              0.807562


In [6]:
# Keep first two columns
pmi_us = pmi_us.iloc[:, [0, 1]]

# Rename columns
pmi_us.columns = ["Date", "PMI - US"]

# Remove first row
pmi_us = pmi_us.iloc[1:]

# Clean spaces
pmi_us["Date"] = pmi_us["Date"].astype(str).str.strip()

# Convert dates
pmi_us["Date"] = pd.to_datetime(
    pmi_us["Date"],
    format="%b %d, %Y",
    errors="coerce"
)

# Remove invalid rows
pmi_us = pmi_us.dropna(subset=["Date"])

# Convert PMI values
pmi_us["PMI - US"] = pd.to_numeric(
    pmi_us["PMI - US"],
    errors="coerce"
)

# Remove NaN values
pmi_us = pmi_us.dropna()

# Reverse chronological order
pmi_us  = pmi_us.sort_values("Date")

# Set index
pmi_us.set_index("Date", inplace=True)

print(pmi_us.head())

            PMI - US
Date                
2016-01-04      48.2
2016-02-01      48.2
2016-03-01      49.5
2016-04-01      51.8
2016-05-02      50.8


In [7]:
# Keep useful columns
pmi_eur = pmi_eur.iloc[:, [0, 1]]

# Rename columns
pmi_eur.columns = ["Date", "PMI - EUR"]

# Convert date format
pmi_eur["Date"] = pd.to_datetime(
    pmi_eur["Date"],
    errors="coerce"
)

# Convert PMI values
pmi_eur["PMI - EUR"] = pd.to_numeric(
    pmi_eur["PMI - EUR"],
    errors="coerce"
)

# Remove invalid rows
pmi_eur = pmi_eur.dropna()

# Set Date as index
pmi_eur.set_index("Date", inplace=True)
pmi_eur.index = pd.to_datetime(pmi_eur.index.date)

print(pmi_eur.head())

            PMI - EUR
2016-04-30       51.5
2016-05-31       52.8
2016-06-30       52.0
2016-07-31       51.7
2016-08-31       52.6


In [8]:
# Keep useful columns
sales_eur = sales_eur.iloc[:, [0, 1]].copy()

# Rename columns
sales_eur.columns = ["Date", "Retail Sales Growth - EUR"]

# Clean date
sales_eur["Date"] = pd.to_datetime(sales_eur["Date"])

# Clean numeric values
sales_eur["Retail Sales Growth - EUR"] = (
    sales_eur["Retail Sales Growth - EUR"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

sales_eur["Retail Sales Growth - EUR"] = pd.to_numeric(
    sales_eur["Retail Sales Growth - EUR"],
    errors="coerce"
)

# Remove invalid rows
sales_eur = sales_eur.dropna()

# Set Date as index
sales_eur.set_index("Date", inplace=True)

print(sales_eur.head())

            Retail Sales Growth - EUR
Date                                 
2016-01-01                   1.916611
2016-04-01                   1.067378
2016-07-01                   1.260783
2016-10-01                   2.389645
2017-01-01                   2.078522


In [9]:
# Keep useful columns
ip_eur = ip_eur.iloc[:, [0, 2]].copy()

# Rename columns
ip_eur.columns = ["Date", "Industrial Production - EUR"]

# Convert date
ip_eur["Date"] = pd.to_datetime(ip_eur["Date"])

# Convert values
ip_eur["Industrial Production - EUR"] = pd.to_numeric(
    ip_eur["Industrial Production - EUR"],
    errors="coerce"
)

# Keep only data from 2016
ip_eur = ip_eur[
    ip_eur["Date"] >= "2016-01-01"
]

# Set Date as index
ip_eur.set_index("Date", inplace=True)

print(ip_eur.head())

            Industrial Production - EUR
Date                                   
2016-01-31                         94.1
2016-02-29                         93.3
2016-03-31                        102.1
2016-04-30                         95.7
2016-05-31                         94.8


In [10]:
# Keep useful columns
ip_us = ip_us.iloc[:, [0, 1]].copy()

# Rename columns
ip_us.columns = ["Date", "Industrial Production - US"]

# Convert date
ip_us["Date"] = pd.to_datetime(ip_us["Date"])

# Clean numeric values
ip_us["Industrial Production - US"] = (
    ip_us["Industrial Production - US"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

# Convert values to numeric
ip_us["Industrial Production - US"] = pd.to_numeric(
    ip_us["Industrial Production - US"],
    errors="coerce"
)

# Remove invalid rows
ip_us = ip_us.dropna()

# Set Date as index
ip_us.set_index("Date", inplace=True)

print(ip_us.head())

            Industrial Production - US
Date                                  
2016-01-01                     99.5607
2016-02-01                     99.0445
2016-03-01                     98.3096
2016-04-01                     98.6421
2016-05-01                     98.4184


In [11]:
# Keep useful columns
conf_us = conf_us.iloc[:, [0, 1]].copy()

# Rename columns
conf_us.columns = ["Date", "Consumer Confidence - US"]

# Convert date
conf_us["Date"] = pd.to_datetime(conf_us["Date"])

# Clean numeric values
conf_us["Consumer Confidence - US"] = (
    conf_us["Consumer Confidence - US"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

# Convert values to numeric
conf_us["Consumer Confidence - US"] = pd.to_numeric(
    conf_us["Consumer Confidence - US"],
    errors="coerce"
)

# Remove invalid rows
conf_us = conf_us.dropna()

# Set Date as index
conf_us.set_index("Date", inplace=True)

print(conf_us.head())

            Consumer Confidence - US
Date                                
2016-01-01                  98.98682
2016-02-01                  98.66404
2016-03-01                  97.91087
2016-04-01                  95.75899
2016-05-01                 101.89190


In [12]:
# Keep useful columns
conf_eur = conf_eur.iloc[:, [0, 1]].copy()

# Rename columns
conf_eur.columns = ["Date", "Consumer Confidence - EUR"]

# Convert date
conf_eur["Date"] = pd.to_datetime(conf_eur["Date"])

# Clean numeric values
conf_eur["Consumer Confidence - EUR"] = (
    conf_eur["Consumer Confidence - EUR"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

# Convert values to numeric
conf_eur["Consumer Confidence - EUR"] = pd.to_numeric(
    conf_eur["Consumer Confidence - EUR"],
    errors="coerce"
)

# Remove invalid rows
conf_eur = conf_eur.dropna()

# Set Date as index
conf_eur.set_index("Date", inplace=True)

print(conf_eur.head())

            Consumer Confidence - EUR
Date                                 
2016-01-01                       -5.2
2016-02-01                       -6.2
2016-03-01                       -7.3
2016-04-01                       -6.9
2016-05-01                       -7.3


In [13]:
# Keep useful columns
nfp_us = nfp_us.iloc[:, [0, 1]].copy()

# Rename columns
nfp_us.columns = ["Date", "Employment Level - US"]

# Convert date
nfp_us["Date"] = pd.to_datetime(nfp_us["Date"])

# Clean numeric values
nfp_us["Employment Level - US"] = (
    nfp_us["Employment Level - US"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

# Convert to numeric
nfp_us["Employment Level - US"] = pd.to_numeric(
    nfp_us["Employment Level - US"],
    errors="coerce"
)

# Remove invalid rows
nfp_us = nfp_us.dropna()

# Set Date as index
nfp_us.set_index("Date", inplace=True)

# Compute monthly employment growth %
nfp_us["Employment Growth - US"] = (
    nfp_us["Employment Level - US"]
    .pct_change() * 100
)

# Keep only growth column
nfp_us = nfp_us[["Employment Growth - US"]]

# Remove first row
nfp_us = nfp_us.iloc[1:]

print(nfp_us.head())

            Employment Growth - US
Date                              
2016-02-01                0.136862
2016-03-01                0.178514
2016-04-01                0.133647
2016-05-01                0.031977
2016-06-01                0.171647


In [14]:
# Keep useful columns
nfp_eur = nfp_eur.iloc[:, [0, 1]].copy()

# Rename columns
nfp_eur.columns = ["Date", "Employment Growth - EUR"]

# Remove first row
nfp_eur = nfp_eur.iloc[1:]

# Clean dates
nfp_eur["Date"] = nfp_eur["Date"].astype(str).str.strip()

# Convert dates
nfp_eur["Date"] = pd.to_datetime(
    nfp_eur["Date"],
    format="%b %d, %Y",
    errors="coerce"
)

# Convert values
nfp_eur["Employment Growth - EUR"] = pd.to_numeric(
    nfp_eur["Employment Growth - EUR"],
    errors="coerce"
)

# Remove invalid rows
nfp_eur = nfp_eur.dropna()


# Reverse chronological order
nfp_eur = nfp_eur.sort_values("Date")


# Set Date as index
nfp_eur.set_index("Date", inplace=True)

print(nfp_eur.head())

            Employment Growth - EUR
Date                               
2016-03-15                    0.003
2016-06-14                    0.003
2016-09-13                    0.004
2016-12-13                    0.002
2017-03-15                    0.003


In [15]:
# Keep useful columns
unemp_us = unemp_us.iloc[:, [0, 1]].copy()

# Rename columns
unemp_us.columns = ["Date", "Unemployment Rate - US"]

# Convert date
unemp_us["Date"] = pd.to_datetime(unemp_us["Date"])

# Keep only data from 2016
unemp_us = unemp_us[
    unemp_us["Date"] >= "2016-01-01"
]

# Convert values
unemp_us["Unemployment Rate - US"] = pd.to_numeric(
    unemp_us["Unemployment Rate - US"],
    errors="coerce"
)

# Remove invalid rows
unemp_us = unemp_us.dropna()

# Set Date as index
unemp_us.set_index("Date", inplace=True)

print(unemp_us.head())

            Unemployment Rate - US
Date                              
2016-01-01                     4.8
2016-02-01                     4.9
2016-03-01                     5.0
2016-04-01                     5.1
2016-05-01                     4.8


In [16]:
# Keep useful columns
unemp_eur = unemp_eur.iloc[:, [0, 1]].copy()

# Rename columns
unemp_eur.columns = ["Date", "Unemployment Rate - EUR"]

# Remove first row
unemp_eur = unemp_eur.iloc[1:]

# Clean dates
unemp_eur["Date"] = unemp_eur["Date"].astype(str).str.strip()

# Convert dates
unemp_eur["Date"] = pd.to_datetime(
    unemp_eur["Date"],
    format="%b %d, %Y",
    errors="coerce"
)

# Convert values
unemp_eur["Unemployment Rate - EUR"] = pd.to_numeric(
    unemp_eur["Unemployment Rate - EUR"],
    errors="coerce"
)

# Remove invalid rows
unemp_eur = unemp_eur.dropna()

# Reverse chronological order
unemp_eur = unemp_eur.sort_values("Date")

# Set Date as index
unemp_eur.set_index("Date", inplace=True)

print(unemp_eur.head())

            Unemployment Rate - EUR
Date                               
2016-03-01                    0.103
2016-04-04                    0.103
2016-04-29                    0.102
2016-05-31                    0.102
2016-07-01                    0.101


In [17]:
# Keep useful columns
wages_us = wages_us.iloc[:, [0, 1]].copy()

# Rename columns
wages_us.columns = ["Date", "Wage Level - US"]

# Convert date
wages_us["Date"] = pd.to_datetime(wages_us["Date"])

# Convert values
wages_us["Wage Level - US"] = pd.to_numeric(
    wages_us["Wage Level - US"],
    errors="coerce"
)

# Remove invalid rows
wages_us = wages_us.dropna()

# Set Date as index
wages_us.set_index("Date", inplace=True)

# Compute wage growth %
wages_us["Wage Growth - US"] = (
    wages_us["Wage Level - US"]
    .pct_change() * 100
)

# Keep only growth column
wages_us = wages_us[["Wage Growth - US"]]

# Remove first row
wages_us = wages_us.iloc[1:]

print(wages_us.head())

            Wage Growth - US
Date                        
2016-02-01          0.093853
2016-03-01          0.281294
2016-04-01          0.327256
2016-05-01          0.093197
2016-06-01          0.186220


In [18]:
# Keep useful columns
wages_eur = wages_eur.iloc[:, [0, 2]].copy()

# Rename columns
wages_eur.columns = ["Date", "Wage Growth - EUR"]

# Convert date
wages_eur["Date"] = pd.to_datetime(wages_eur["Date"])

# Keep only data from 2016
wages_eur = wages_eur[
    wages_eur["Date"] >= "2016-01-01"
]

# Convert values
wages_eur["Wage Growth - EUR"] = pd.to_numeric(
    wages_eur["Wage Growth - EUR"],
    errors="coerce"
)

# Remove invalid rows
wages_eur = wages_eur.dropna()

# Set Date as index
wages_eur.set_index("Date", inplace=True)

print(wages_eur.head())

            Wage Growth - EUR
Date                         
2016-03-31               1.31
2016-06-30               1.34
2016-09-30               1.42
2016-12-31               1.40
2017-03-31               1.54


In [19]:
# Keep useful columns
cpi_us = cpi_us.iloc[:, [0, 1]].copy()

# Rename columns
cpi_us.columns = ["Date", "CPI Level - US"]

# Convert date
cpi_us["Date"] = pd.to_datetime(cpi_us["Date"])

# Convert values
cpi_us["CPI Level - US"] = pd.to_numeric(
    cpi_us["CPI Level - US"],
    errors="coerce"
)

# Remove invalid rows
cpi_us = cpi_us.dropna()

# Set Date as index
cpi_us.set_index("Date", inplace=True)

# Compute inflation rate %
cpi_us["Inflation Rate - US"] = (
    cpi_us["CPI Level - US"]
    .pct_change() * 100
)

# Keep only inflation column
cpi_us = cpi_us[["Inflation Rate - US"]]

# Remove first row
cpi_us = cpi_us.iloc[1:]

print(cpi_us.head())

            Inflation Rate - US
Date                           
2016-02-01            -0.132968
2016-03-01             0.313480
2016-04-01             0.383065
2016-05-01             0.236410
2016-06-01             0.277596


In [20]:
# Keep useful columns
cpi_eur = cpi_eur.iloc[:, [0, 1]].copy()

# Rename columns
cpi_eur.columns = ["Date", "CPI Level - EUR"]

# Convert date
cpi_eur["Date"] = pd.to_datetime(cpi_eur["Date"])

# Clean numeric values
cpi_eur["CPI Level - EUR"] = (
    cpi_eur["CPI Level - EUR"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

# Convert values
cpi_eur["CPI Level - EUR"] = pd.to_numeric(
    cpi_eur["CPI Level - EUR"],
    errors="coerce"
)

# Remove invalid rows
cpi_eur = cpi_eur.dropna()

# Set Date as index
cpi_eur.set_index("Date", inplace=True)

# Compute inflation rate %
cpi_eur["Inflation Rate - EUR"] = (
    cpi_eur["CPI Level - EUR"]
    .pct_change() * 100
)

# Keep only inflation column
cpi_eur = cpi_eur[["Inflation Rate - EUR"]]

# Remove first row
cpi_eur = cpi_eur.iloc[1:]

print(cpi_eur.head())

            Inflation Rate - EUR
Date                            
2016-02-01              0.169846
2016-03-01              1.239077
2016-04-01              0.219016
2016-05-01              0.411364
2016-06-01              0.179234


In [21]:
# Keep useful columns
core_cpi_us = core_cpi_us.iloc[:, [0, 1]].copy()

# Rename columns
core_cpi_us.columns = ["Date", "Core CPI Level - US"]

# Convert date
core_cpi_us["Date"] = pd.to_datetime(core_cpi_us["Date"])

# Clean numeric values
core_cpi_us["Core CPI Level - US"] = (
    core_cpi_us["Core CPI Level - US"]
    .astype(str)
    .str.replace(";", "", regex=False)
)

# Convert values
core_cpi_us["Core CPI Level - US"] = pd.to_numeric(
    core_cpi_us["Core CPI Level - US"],
    errors="coerce"
)

# Remove invalid rows
core_cpi_us = core_cpi_us.dropna()

# Set Date as index
core_cpi_us.set_index("Date", inplace=True)

# Compute core inflation rate %
core_cpi_us["Core Inflation Rate - US"] = (
    core_cpi_us["Core CPI Level - US"]
    .pct_change() * 100
)

# Keep only inflation column
core_cpi_us = core_cpi_us[["Core Inflation Rate - US"]]

# Remove first row
core_cpi_us = core_cpi_us.iloc[1:]

print(core_cpi_us.head())

            Core Inflation Rate - US
Date                                
2016-02-01                  0.226572
2016-03-01                  0.164148
2016-04-01                  0.259441
2016-05-01                  0.237679
2016-06-01                  0.163067


In [22]:
# Keep useful columns
core_cpi_eur = core_cpi_eur.iloc[:, [0, 1]].copy()

# Rename columns
core_cpi_eur.columns = ["Date", "Core CPI Level - EUR"]

# Convert date
core_cpi_eur["Date"] = pd.to_datetime(core_cpi_eur["Date"])

# Convert values
core_cpi_eur["Core CPI Level - EUR"] = pd.to_numeric(
    core_cpi_eur["Core CPI Level - EUR"],
    errors="coerce"
)

# Remove invalid rows
core_cpi_eur = core_cpi_eur.dropna()

# Set Date as index
core_cpi_eur.set_index("Date", inplace=True)

# Compute core inflation rate %
core_cpi_eur["Core Inflation Rate - EUR"] = (
    core_cpi_eur["Core CPI Level - EUR"]
    .pct_change() * 100
)

# Keep only inflation column
core_cpi_eur = core_cpi_eur[["Core Inflation Rate - EUR"]]

# Remove first row
core_cpi_eur = core_cpi_eur.iloc[1:]

print(core_cpi_eur.head())

            Core Inflation Rate - EUR
Date                                 
2016-02-01                   0.407458
2016-03-01                   1.574029
2016-04-01                   0.217918
2016-05-01                   0.289925
2016-06-01                   0.048181
